# Computational Theory
**Author** Sarah O'Connor - G00423847

### Introduction

SHA-256 is a cryptographic hash function defined in the Secure Hash Standard (FIPS 180-4) and is widely used in security applications such as data integrity checks, digital signatures, and password storage. It operates on fixed-size 512-bit blocks and produces a 256-bit (32-byte) hash value.

This assignment looks at the internal structure of SHA-256 by recreating its core components in Python. Rather than treating SHA-256 as a black box, the problems focus on the bit-level operations, constants, padding rules, and compression function that together define the algorithm.

The notebook progresses from basic logical functions and rotations, through constant generation and message padding, to a full implementation of the SHA-256 compression step. Finally, a password-cracking exercise demonstrates why SHA-256 is unsuitable for password storage and motivates the use of modern password hashing schemes.

All implementations closely follow the Secure Hash Standard and are tested against known reference values to ensure correctness.


In [2438]:
# import statements
import numpy as np
import hashlib

---

## Problem 1: Binary Words and Operations
Implement the following functions in Python. Use numpy to ensure that all variables and values are treated as 32-bit integers. These functions are defined in the Secure Hash Standard 

1. `Parity(x, y, z)`
2. `Ch(x, y, z)`
3. `Maj(x, y, z)`
4. `Sigma0(x)` - written as $\Sigma_0^{\{256\}}(x)$ in the standard.
5. `Sigma1(x)` - written as $\Sigma_1^{\{256\}}(x)$ in the standard.
6. `sigma0(x)` - written as $\sigma_0^{\{256\}}(x)$ in the standard.
7. `sigma1(x)` - written as $\sigma_1^{\{256\}}(x)$ in the standard.


### 1.1 Parity(x, y, z)

The Parity function is defined in the Secure Hash Standard, which can be found in Section 4.1.1, or page 10, of the [Secure Hash Standard (FIPS PUB 180-4)](https://github.com/ianmcloughlin/computational-theory/blob/main/materials/secure-hash-standard.pdf). 

**What it does**  
Parity operates bit-by-bit across three 32-bit words. For each bit position:
- if an odd number of the input bits are 1 (e.g. 1 or 3 ones), the output bit is 1  
- if an even number of the input bits are 1 (e.g. 0 or 2 ones), the output bit is 0  

**Why it's useful in hashing**  
Cryptographic hash compression functions repeatedly mix internal state using simple operations (XOR, AND, NOT, rotations, additions). Parity is useful because:
- it is fast, asit relies on bitwise operations,
- it is balanced, as each output bit depends on the corresponding bits of all three inputs,
- it helps provide diffusion, as small changes in inputs can flip output bits. This is especially true after many rounds combined with rotations and additions.

**How it is implemented below**  
In Python, integers can grow beyond 32 bits, but SHA-style functions require 32-bit word behaviour. I  casted each input to `numpy.uint32`, to ensure:
- operations wrap around at 32 bits,
- bitwise operators behave like 32-bit machine arithmetic.

The implementation is a direct translation of the definition: `x ^ y ^ z`.


In [2439]:
def Parity(x, y, z):
    """
    Compute the Parity logical function.

    Parity(x, y, z) = x XOR y XOR z, applied bitwise over 32-bit words.
    For each bit position, the result is 1 iff an odd number of input bits are 1.

    Args:
        x, y, z: Values interpreted as 32-bit words.

    Returns:
        numpy.uint32: The 32-bit result of x ^ y ^ z.
    """
    
    # Ensure inputs are treated as 32-bit unsigned integers
    x, y, z = np.uint32(x), np.uint32(y), np.uint32(z)
    
    # Return the bitwise XOR of the three inputs
    return x ^ y ^ z

#### Parity Testing

The Parity function is tested using three cases, with printed output to make the results easy to verify by inspection.

- **Test 1 (all zeros)** checks the baseline behaviour of the function. If all input bits are zero, the output must also be zero.
- **Test 2 (small alternating patterns)** uses short binary values that can be checked by hand. This confirms that the XOR parity rule is applied correctly at each bit position.
- **Test 3 (typical 32-bit words)** uses larger values more representative of real hash state words. The expected result is computed using Python’s built-in XOR operator, ensuring the implementation behaves correctly for realistic inputs.

The tests print both the expected and actual values in binary form, making it easy to visually confirm correctness and spot any bit-level errors.


In [2440]:
def test_parity():
    print("Testing Parity function...")
    print("-" * 40)
    
    # Test case 1: All zeros
    x = np.uint32(0b0000)
    y = np.uint32(0b0000)
    z = np.uint32(0b0000)
    expected = np.uint32(0b0000) 
    result = Parity(x, y, z)
    
    print("Test 1:")
    print(" x =", bin(x))
    print(" y =", bin(y))
    print(" z =", bin(z))
    print("Expected:", bin(expected))
    print("Actual:  ", bin(result))
    
    if expected == result:
        print("TEST 1: PASSED")
    else:
        print("TEST 1: FAILED")   
    print()
       
    # Test case 2: Alternating bits
    x = np.uint32(0b1010)
    y = np.uint32(0b1100)
    z = np.uint32(0b0111)
    expected = np.uint32(0b0001) 
    result = Parity(x, y, z)
    
    print("Test 2:")
    print(" x =", bin(x))
    print(" y =", bin(y))
    print(" z =", bin(z))
    print("Expected:", bin(expected))
    print("Actual:  ", bin(result))   
     
    if expected == result:
        print("TEST 2: PASSED")
    else:
        print("TEST 2: FAILED")
    print()
    
    # Test case 3: Typical words
    x = np.uint32(0x12345678)
    y = np.uint32(0x0f0f0f0f)
    z = np.uint32(0xaaaaaaaa)
    expected = np.uint32(x ^ y ^ z)
    result = Parity(x, y, z)
    
    print("Test 3:")
    print(" x =", bin(x))
    print(" y =", bin(y))
    print(" z =", bin(z))
    print("Expected:", bin(expected))
    print("Actual:  ", bin(result))
    
    if expected == result:
        print("TEST 3: PASSED")
    else:
        print("TEST 3: FAILED")
    print()
    

In [2441]:
test_parity()

Testing Parity function...
----------------------------------------
Test 1:
 x = 0b0
 y = 0b0
 z = 0b0
Expected: 0b0
Actual:   0b0
TEST 1: PASSED

Test 2:
 x = 0b1010
 y = 0b1100
 z = 0b111
Expected: 0b1
Actual:   0b1
TEST 2: PASSED

Test 3:
 x = 0b10010001101000101011001111000
 y = 0b1111000011110000111100001111
 z = 0b10101010101010101010101010101010
Expected: 0b10110111100100011111001111011101
Actual:   0b10110111100100011111001111011101
TEST 3: PASSED



---

### 1.2 Ch(x, y, z)
The **Ch** (“choose”) function is defined in the Secure Hash Standard (FIPS 180-4, Section 4.1.2 / page 10). It is one of the main logical functions used in SHA-256 during the compression rounds.

**What it does**  
Ch operates bit-by-bit across three 32-bit words. For each bit position:
- if the bit in `x` is 1, the output bit is taken from `y`
- if the bit in `x` is 0, the output bit is taken from `z`

**Why it's useful in hashing**  
`Ch` introduces non-linearity into the compression function. Rather than combining inputs symmetrically (like XOR), it conditionally selects bits, which helps SHA-256 mix state in a way that is hard to reverse. A small change in `x` can cause many output bits to switch between coming from `y` and `z`.

**How it is implemented below**  
The `Ch` function can be read directly from the bit logic:
- `x & y` keeps the bits from `y` only where `x` has 1s.
- `~x & z` keeps the bits from `z` only where `x` has 0s (because `~x` flips the mask).
- XOR (`^`) then combines these two parts. They don’t overlap (a bit can’t be selected from both `y` and `z` at the same time), so XOR works as a sort of merge.


In [2442]:
def Ch(x, y, z):
    """
    Compute the Choose (Ch) function used in SHA-256.

    For each bit position, if the bit of x is 1 the output bit comes from y,
    otherwise it comes from z.

    Args:
        x, y, z: Values interpreted as 32-bit words.

    Returns:
        numpy.uint32: The 32-bit result of the choose operation.
    """
    
    # Ensure inputs are treated as 32-bit unsigned integers
    x, y, z = np.uint32(x), np.uint32(y), np.uint32(z)
    
    # XOR operation to select bits from y or z based on x
    return (x & y) ^ (~x & z)

#### Ch Testing
The tests for `Ch(x, y, z)` are designed to show the “choose” behaviour clearly using small binary inputs that can be checked by hand.

- **Test 1 (x = 0)**: when all bits in `x` are 0, the function should select all bits from `z`, so the result must equal `z`.
- **Test 2 (x = 1)**: when all bits in `x` are 1, the function should select all bits from `y`, so the result must equal `y`.
- **Test 3 (mixed mask)**: uses a patterned `x` so that some output bits come from `y` and others from `z`. This confirms the selection is happening bit-by-bit rather than treating the inputs as whole numbers.

The tests print expected vs actual values in binary to make any bit-level mistake obvious.

In [2443]:
def test_ch():
    print("Testing Ch function...")
    print("-" * 40)

    
    # Test case 1: x selects entirely from z
    x = np.uint32(0b0000)
    y = np.uint32(0b1111)
    z = np.uint32(0b0101)
    expected = np.uint32(0b0101)   # x = 0 → take all bits from z
    result = Ch(x, y, z)
    
    print("Test 1:")
    print(" x =", bin(x))
    print(" y =", bin(y))
    print(" z =", bin(z))
    print("Expected:", bin(expected))
    print("Actual:  ", bin(result))
    
    if expected == result:
        print("TEST 1: PASSED")
    else:
        print("TEST 1: FAILED")
    print()
    
    # Test case 2: x selects entirely from y
    x = np.uint32(0b1111)
    y = np.uint32(0b1010)
    z = np.uint32(0b0101)
    expected = np.uint32(0b1010)   # x = 1 → take all bits from y
    result = Ch(x, y, z)
    
    print("Test 2:")
    print(" x =", bin(x))
    print(" y =", bin(y))
    print(" z =", bin(z))
    print("Expected:", bin(expected))
    print("Actual:  ", bin(result))
    
    if expected == result:
        print("TEST 2: PASSED")
    else:
        print("TEST 2: FAILED")
    print()
    
    # Test case 3: Mixed selection
    x = np.uint32(0b1010)
    y = np.uint32(0b1100)
    z = np.uint32(0b0111)
    expected = np.uint32(0b1101)
    result = Ch(x, y, z)
    
    print("Test 3:")
    print(" x =", bin(x))
    print(" y =", bin(y))
    print(" z =", bin(z))
    print("Expected:", bin(expected))
    print("Actual:  ", bin(result))
    
    if expected == result:
        print("TEST 3: PASSED")
    else:
        print("TEST 3: FAILED")
    print()


In [2444]:
test_ch()

Testing Ch function...
----------------------------------------
Test 1:
 x = 0b0
 y = 0b1111
 z = 0b101
Expected: 0b101
Actual:   0b101
TEST 1: PASSED

Test 2:
 x = 0b1111
 y = 0b1010
 z = 0b101
Expected: 0b1010
Actual:   0b1010
TEST 2: PASSED

Test 3:
 x = 0b1010
 y = 0b1100
 z = 0b111
Expected: 0b1101
Actual:   0b1101
TEST 3: PASSED



---

### 1.3 Maj(x, y, z)
The Maj (“majority”) function is defined in the Secure Hash Standard (FIPS 180-4, Section 4.1.2 / page 10). 

**What it does**  
Maj operates bit-by-bit across three 32-bit words. For each bit position:
- the output bit is 1 if at least two of the three input bits are 1
- otherwise, the output bit is 0

The output bit represents the majority value of the corresponding bits in `x`, `y`, and `z`.

**Why it's useful in hashing**  
`Maj` helps to mix the internal state but keep balance. Each output bit depends on all three inputs, but no single input fully controls the result. This symmetry helps distribute influence evenly across the state and contributes to diffusion when combined with rotations and modular addition.

**How it is implemented below**  
The majority behaviour can be expressed using bitwise operations:
- `x & y` selects bits where both `x` and `y` are 1
- `x & z` selects bits where both `x` and `z` are 1
- `y & z` selects bits where both `y` and `z` are 1  

If at least two inputs have a 1 at a given bit position, one of these expressions will capture it. XOR is then used to combine the results into the final output.

As with the other functions, inputs are cast to `np.uint32` so that all operations behave like 32-bit word arithmetic.

In [2445]:
def Maj(x, y, z):
    """
    Compute the Majority (Maj) function used in SHA-256.

    For each bit position, the output bit is 1 if at least two of the
    corresponding bits in x, y, and z are 1; otherwise it is 0.

    Args:
        x, y, z: Values interpreted as 32-bit words.

    Returns:
        numpy.uint32: The 32-bit result of the majority operation.
    """
    # Ensure inputs are treated as 32-bit unsigned integers
    x, y, z = np.uint32(x), np.uint32(y), np.uint32(z)
    
    # Return the majority function result
    return (x & y) ^ (x & z) ^ (y & z)


#### Maj Testing
The tests for `Maj(x, y, z)` are written to show the majority behaviour using clear and easily followed examples.

- **Test 1 (all zeros)** checks the baseline case. If all input bits are 0, the majority at every position must also be 0.
- **Test 2 (all ones)** checks the opposite of test 1. If all input bits are 1, the majority is always 1.
- **Test 3 (mixed bits)** uses patterned inputs so that some bit positions have a majority of 1s while others do not. This confirms that the function operates correctly on a bit-by-bit basis rather than treating inputs as whole values.


In [2446]:
def test_maj():
    print("Testing Maj function...")
    print("-" * 40)

    # Test case 1: All zeros
    x = np.uint32(0b0000)
    y = np.uint32(0b0000)
    z = np.uint32(0b0000)
    expected = np.uint32(0b0000)
    result = Maj(x, y, z)
    
    print("Test 1:")
    print(" x =", bin(x))
    print(" y =", bin(y))
    print(" z =", bin(z))
    print("Expected:", bin(expected))
    print("Actual:  ", bin(result))
    
    if expected == result:
        print("TEST 1: PASSED")
    else:
        print("TEST 1: FAILED")
    print()
    
    # Test case 2: All ones
    x = np.uint32(0b1111)
    y = np.uint32(0b1111)
    z = np.uint32(0b1111)
    expected = np.uint32(0b1111)
    result = Maj(x, y, z)
    
    print("Test 2:")
    print(" x =", bin(x))
    print(" y =", bin(y))
    print(" z =", bin(z))
    print("Expected:", bin(expected))
    print("Actual:  ", bin(result))
    
    if expected == result:
        print("TEST 2: PASSED")
    else:
        print("TEST 2: FAILED")
    print()
    
    # Test case 3: Mixed bits
    x = np.uint32(0b1010)
    y = np.uint32(0b1100)
    z = np.uint32(0b1001)
    expected = np.uint32(0b1000)
    result = Maj(x, y, z)
    
    print("Test 3:")
    print(" x =", bin(x))
    print(" y =", bin(y))
    print(" z =", bin(z))
    print("Expected:", bin(expected))
    print("Actual:  ", bin(result))
    
    if expected == result:
        print("TEST 3: PASSED")
    else:
        print("TEST 3: FAILED")
    print()


In [2447]:
test_maj()

Testing Maj function...
----------------------------------------
Test 1:
 x = 0b0
 y = 0b0
 z = 0b0
Expected: 0b0
Actual:   0b0
TEST 1: PASSED

Test 2:
 x = 0b1111
 y = 0b1111
 z = 0b1111
Expected: 0b1111
Actual:   0b1111
TEST 2: PASSED

Test 3:
 x = 0b1010
 y = 0b1100
 z = 0b1001
Expected: 0b1000
Actual:   0b1000
TEST 3: PASSED



---

### 1.4 ROTR

SHA-256 often uses bit rotations, particularly in the Σ (Sigma) and σ (sigma) functions defined later. To avoid repeating the same rotation logic multiple times, I defined a small helper function for rotate right (ROTR).

**What it does**  
A right rotation shifts the bits of a 32-bit word to the right by a given number of positions. Bits that fall off the right-hand side are wrapped around and are appended on the left-hand side. This differs from a right shift, which discards bits instead of keeping them.

**Why it is needed**  
Rotations are a key part of SHA-256 because they:
- preserve all bits of the input value,
- redistribute bits to different positions,
- help provide diffusion when combined with XOR and addition.

**How it is implemented below**  
The function combines a right shift and a left shift, then merges the results using bitwise OR. The rotation count is masked with `31` to ensure it stays within the valid range for a 32-bit word. Inputs are cast to `np.uint32` so that the result behaves like a 32-bit word.

In [2448]:
def ROTR(x, n):
    """
    Rotate a 32-bit word to the right by n bits.

    Unlike a right shift, rotation preserves all bits by wrapping the bits
    that fall off the right-hand side back around to the left.

    Args:
        x: Value interpreted as a 32-bit word.
        n: Number of bit positions to rotate.

    Returns:
        numpy.uint32: The rotated 32-bit result.
    """
    # Ensure input is treated as a 32-bit unsigned integer
    x = np.uint32(x)
    
    # ensure rotation count is within 0–31
    n &= 31
    if n == 0:
        return x
    
    return np.uint32((x >> n) | (x << (32 - n)))

---

### 1.5 Sigma0(x)
The Σ0 (Sigma0) function is defined in the Secure Hash Standard (FIPS 180-4, Section 4.1.2 / page 10). It is one of the two “big sigma" functions used in the main compression loop of SHA-256.

**What it does**  
Sigma0 operates on a single 32-bit word and produces a new 32-bit value by combining several rotated versions of the input. It applies three different right rotations to the same input word and XORs the results together.

**Why it's useful in hashing**  
The purpose of Sigma functions in SHA-256 is to provide diffusion. By rotating the same word by different amounts and combining the results, each output bit depends on multiple input bits from different positions. This helps ensure that small changes in the input quickly spread throughout the internal state during the compression rounds.

**How it is implemented below**  
Sigma0 is implemented by calling the previously defined `ROTR` helper with three fixed rotation amounts (2, 13, and 22 bits), then XORing the rotated values together.

In [2449]:
def Sigma0(x):
    """
    Compute the Sigma0 (Σ0) function used in SHA-256.

    Sigma0 applies three different right rotations to a 32-bit word and
    combines the results using XOR to produce a diffused output.

    Args:
        x: Value interpreted as a 32-bit word.

    Returns:
        numpy.uint32: The 32-bit Sigma0 transformation of x.
    """
    return np.uint32(ROTR(x, 2) ^ ROTR(x, 13) ^ ROTR(x, 22))

#### Sigma0 Testing

- **Test 1 (all zeros)** verifies the baseline behaviour. Rotating zero always produces zero, so the output must also be zero.
- **Test 2 (single bit set)** tracks how a single 1-bit moves under each rotation. This makes it easy to reason about the expected output by hand.
- **Test 3 (typical 32-bit value)** uses a realistic input value and compares the result against a direct expression using the same rotation logic, confirming correct behaviour for general inputs.

In [2450]:
def test_Sigma0():
    print("Testing Sigma0 function...")
    print("-" * 40)

    # Test case 1: All zeros
    x = np.uint32(0b0000)
    expected = np.uint32(0b0000)
    result = Sigma0(x)
    
    print("Test 1:")
    print(" x =", bin(x))
    print("Expected:", bin(expected))
    print("Actual:  ", bin(result))
    
    if expected == result:
        print("TEST 1: PASSED")
    else:
        print("TEST 1: FAILED")
    print()
    
    # Test case 2: Single bit set
    x = np.uint32(0b0001)
    # ROTR(1,2)  -> bit moves to position 30
    # ROTR(1,13) -> bit moves to position 19
    # ROTR(1,22) -> bit moves to position 10
    expected = np.uint32((1 << 30) ^ (1 << 19) ^ (1 << 10))
    result = Sigma0(x)
    
    print("Test 2:")
    print(" x =", bin(x))
    print("Expected:", bin(expected))
    print("Actual:  ", bin(result))
    
    if expected == result:
        print("TEST 2: PASSED")
    else:
        print("TEST 2: FAILED")
    print()
    
    # Test case 3: Random 32-bit value
    x = np.uint32(0x12345678)
    expected = np.uint32(ROTR(x, 2) ^ ROTR(x, 13) ^ ROTR(x, 22))
    result = Sigma0(x)
    
    print("Test 3:")
    print(" x =", hex(x))
    print("Expected:", hex(expected))
    print("Actual:  ", hex(result))
    
    if expected == result:
        print("TEST 3: PASSED")
    else:
        print("TEST 3: FAILED")
    print()


In [2451]:
test_Sigma0()

Testing Sigma0 function...
----------------------------------------
Test 1:
 x = 0b0
Expected: 0b0
Actual:   0b0
TEST 1: PASSED

Test 2:
 x = 0b1
Expected: 0b1000000000010000000010000000000
Actual:   0b1000000000010000000010000000000
TEST 2: PASSED

Test 3:
 x = 0x12345678
Expected: 0x66146474
Actual:   0x66146474
TEST 3: PASSED



---

### 1.6 Sigma1(x)

The **Σ1 (Sigma1)** function is defined in the Secure Hash Standard (FIPS 180-4, Section 4.1.2 / page 10). Like Sigma0, it is one of the two “big sigma” functions used in the main compression loop of SHA-256.

**What it does**  
Sigma1 takes a single 32-bit word and produces a new 32-bit value by XORing together three rotated versions of the input. It uses three different right rotations, which redistributes the bits before combining them.

**Why it's useful in hashing**  
Sigma1 contributes to diffusion in the SHA-256 compression function. Rotating by multiple different offsets ensures that bits from the input word influence many different bit positions in the result.

**How it is implemented below**  
Sigma1 is implemented by calling the `ROTR` helper with three fixed rotation amounts (6, 11, and 25 bits), then XORing those rotated values together. The result is returned as a `np.uint32` to keep 32-bit word behaviour consistent throughout the SHA-256 implementation.


In [2452]:
def Sigma1(x):
    """
    Compute the Sigma1 (Σ1) function used in SHA-256.

    Sigma1 applies three different right rotations to a 32-bit word and
    combines the results using XOR.

    Args:
        x: Value interpreted as a 32-bit word.

    Returns:
        numpy.uint32: The 32-bit Sigma1 transformation of x.
    """
    return np.uint32(ROTR(x, 6) ^ ROTR(x, 11) ^ ROTR(x, 25))

#### Sigma1 Testing

- **Test 1 (all zeros)** verifies baseline behaviour. Rotating zero always produces zero, so the output must be zero.
- **Test 2 (single bit set)** tracks how one 1-bit moves under each rotation. This makes it straightforward to build the expected output manually and confirm the XOR combination.
- **Test 3 (typical 32-bit value)** uses a realistic input and compares the output against the same expression written out directly, confirming correct behaviour for general inputs.

In [2453]:
def test_Sigma1():
    print("Testing Sigma1 function...")
    print("-" * 40)
    
    # Test case 1: All zeros
    x = np.uint32(0b0000)
    expected = np.uint32(0b0000)
    result = Sigma1(x)
    
    print("Test 1:")
    print(" x =", bin(x))
    print("Expected:", bin(expected))
    print("Actual:  ", bin(result))
    
    if expected == result:
        print("TEST 1: PASSED")
    else:
        print("TEST 1: FAILED")
    print()
    
    # Test case 2: Single bit set
    x = np.uint32(0b0001)
    # ROTR(1,6)  -> bit moves to position 26
    # ROTR(1,11) -> bit moves to position 21
    # ROTR(1,25) -> bit moves to position 7
    expected = np.uint32((1 << 26) ^ (1 << 21) ^ (1 << 7))
    result = Sigma1(x)
    
    print("Test 2:")
    print(" x =", bin(x))
    print("Expected:", bin(expected))
    print("Actual:  ", bin(result))
    
    if expected == result:
        print("TEST 2: PASSED")
    else:
        print("TEST 2: FAILED")
    print()
    
    # Test case 3: Typical 32-bit value
    x = np.uint32(0x12345678)
    expected = np.uint32(ROTR(x, 6) ^ ROTR(x, 11) ^ ROTR(x, 25))
    result = Sigma1(x)
    
    print("Test 3:")
    print(" x =", hex(x))
    print("Expected:", hex(expected))
    print("Actual:  ", hex(result))
    
    if expected == result:
        print("TEST 3: PASSED")
    else:
        print("TEST 3: FAILED")
    print()


In [2454]:
test_Sigma1()

Testing Sigma1 function...
----------------------------------------
Test 1:
 x = 0b0
Expected: 0b0
Actual:   0b0
TEST 1: PASSED

Test 2:
 x = 0b1
Expected: 0b100001000000000000010000000
Actual:   0b100001000000000000010000000
TEST 2: PASSED

Test 3:
 x = 0x12345678
Expected: 0x3561abda
Actual:   0x3561abda
TEST 3: PASSED



---

### 1.7 sigma0(x)

The σ0 (sigma0) function is defined in the Secure Hash Standard (FIPS 180-4, Section 4.1.2 / page 10). It is one of the two “small sigma” functions used when building the message schedule for SHA-256.

**What it does**  
sigma0 operates on a single 32-bit word and combines rotated and shifted versions of the input. It applies two right rotations and one logical right shift, then XORs the results together.

**Why it's useful in hashing**  
The small sigma functions are used to expand the original message into a larger message schedule. The inclusion of a right shift (which discards bits rather than wrapping them) helps break up bit patterns and ensures that earlier message words influence later ones in a non-reversible way.

This contributes to diffusion across the message schedule before the compression rounds even begin.

**How it is implemented below**  
sigma0 is implemented by rotating the input right by 7 and 18 bits, shifting it right by 3 bits, and XORing the three results together. The final result is returned as a `np.uint32` to maintain 32-bit word behaviour.

In [2455]:
def sigma0(x):
    """
    Compute the sigma0 (σ0) function used in the SHA-256 message schedule.

    sigma0 combines two right rotations and one logical right shift to
    produce a diffused 32-bit output.

    Args:
        x: Value interpreted as a 32-bit word.

    Returns:
        numpy.uint32: The 32-bit sigma0 transformation of x.
    """
    return np.uint32(ROTR(x, 7) ^ ROTR(x, 18) ^ (x >> 3))

#### sigma0 Testing

- **Test 1 (all zeros)** confirms baseline behaviour. Rotations and shifts of zero always produce zero.
- **Test 2 (single bit set)** shows how rotations preserve the bit by moving it to new positions, while the right shift removes it entirely.
- **Test 3 (arbitrary 32-bit value)** checks correct behaviour for a typical input by comparing against the direct expression.

In [2456]:
def test_sigma0():
    print("Testing sigma0 function...")
    print("-" * 40)
    
    # Test case 1: All zeros
    x = np.uint32(0b0000)
    expected = np.uint32(0b0000)
    result = sigma0(x)
    
    print("Test 1:")
    print(" x =", bin(x))
    print("Expected:", bin(expected))
    print("Actual:  ", bin(result))
    
    if expected == result:
        print("TEST 1: PASSED")
    else:
        print("TEST 1: FAILED")
    print()
    
    # Test case 2: Single bit set
    x = np.uint32(0b0001)
    # ROTR(1,7)  -> bit moves to position 25
    # ROTR(1,18) -> bit moves to position 14
    # SHR(1,3)   -> becomes 0
    expected = np.uint32((1 << 25) ^ (1 << 14))
    result = sigma0(x)
    
    print("Test 2:")
    print(" x =", bin(x))
    print("Expected:", bin(expected))
    print("Actual:  ", bin(result))
    
    if expected == result:
        print("TEST 2: PASSED")
    else:
        print("TEST 2: FAILED")
    print()
    
    # Test case 3: Arbitrary 32-bit value
    x = np.uint32(0x12345678)
    expected = np.uint32(ROTR(x, 7) ^ ROTR(x, 18) ^ (x >> 3))
    result = sigma0(x)
    
    print("Test 3:")
    print(" x =", hex(x))
    print("Expected:", hex(expected))
    print("Actual:  ", hex(result))
    
    if expected == result:
        print("TEST 3: PASSED")
    else:
        print("TEST 3: FAILED")
    print()

In [2457]:
test_sigma0()

Testing sigma0 function...
----------------------------------------
Test 1:
 x = 0b0
Expected: 0b0
Actual:   0b0
TEST 1: PASSED

Test 2:
 x = 0b1
Expected: 0b10000000000100000000000000
Actual:   0b10000000000100000000000000
TEST 2: PASSED

Test 3:
 x = 0x12345678
Expected: 0xe7fce6ee
Actual:   0xe7fce6ee
TEST 3: PASSED



---

### 1.8 sigma1(x)

The σ1 (sigma1) function is defined in the Secure Hash Standard (FIPS 180-4, Section 4.1.2 / page 10). Along with sigma0, it is used to generate the expanded message schedule for SHA-256.

**What it does**  
sigma1 operates on a single 32-bit word and produces a new value by combining two right rotations and one logical right shift. The results are XORed together to produce the final output.

**Why it's useful in hashing**  
sigma1 ensures that bits from earlier message words continue to influence later words in the message schedule. The mix of rotations (which preserve bits) and shifts (which discard bits) helps prevent simple relationships between input and expanded message words, strengthening diffusion before the compression function is applied.

**How it is implemented below**  
sigma1 is implemented by rotating the input right by 17 and 19 bits, shifting it right by 10 bits, and XORing the results together. The result is returned as a 32-bit word using `np.uint32`.


In [2458]:
def sigma1(x):
    """
    Compute the sigma1 (σ1) function used in the SHA-256 message schedule.

    sigma1 combines two right rotations and one logical right shift to
    produce a diffused 32-bit output.

    Args:
        x: Value interpreted as a 32-bit word.

    Returns:
        numpy.uint32: The 32-bit sigma1 transformation of x.
    """
    return np.uint32(ROTR(x, 17) ^ ROTR(x, 19) ^ (x >> 10))

#### sigma1 Testing

- **Test 1 (all zeros)** confirms baseline behaviour.
- **Test 2 (single bit set)** demonstrates how the bit moves under rotations and is removed by the right shift.
- **Test 3 (random 32-bit value)** confirms correct behaviour for general inputs by comparing against the same expression written explicitly.

In [2459]:
def test_sigma1():
    print("Testing sigma1 function...")
    print("-" * 40)
    
    # Test case 1: All zeros
    x = np.uint32(0b0000)
    expected = np.uint32(0b0000)
    result = sigma1(x)
    
    print("Test 1:")
    print(" x =", bin(x))
    print("Expected:", bin(expected))
    print("Actual:  ", bin(result))
    
    if expected == result:
        print("TEST 1: PASSED")
    else:
        print("TEST 1: FAILED")
    print()
    
    # Test case 2: Single bit set
    x = np.uint32(0b0001)
    # ROTR(1,17) -> bit moves to position 15
    # ROTR(1,19) -> bit moves to position 13
    # SHR(1,10)  -> becomes 0
    expected = np.uint32((1 << 15) ^ (1 << 13))
    result = sigma1(x)
    
    print("Test 2:")
    print(" x =", bin(x))
    print("Expected:", bin(expected))
    print("Actual:  ", bin(result))
    
    if expected == result:
        print("TEST 2: PASSED")
    else:
        print("TEST 2: FAILED")
    print()
    
    # Test case 3: Random 32-bit value
    x = np.uint32(0x12345678)
    expected = np.uint32(ROTR(x, 17) ^ ROTR(x, 19) ^ (x >> 10))
    result = sigma1(x)
    
    print("Test 3:")
    print(" x =", hex(x))
    print("Expected:", hex(expected))
    print("Actual:  ", hex(result))
    
    if expected == result:
        print("TEST 3: PASSED")
    else:
        print("TEST 3: FAILED")
    print()


In [2460]:
test_sigma1()

Testing sigma1 function...
----------------------------------------
Test 1:
 x = 0b0
Expected: 0b0
Actual:   0b0
TEST 1: PASSED

Test 2:
 x = 0b1
Expected: 0b1010000000000000
Actual:   0b1010000000000000
TEST 2: PASSED

Test 3:
 x = 0x12345678
Expected: 0xa1f78649
Actual:   0xa1f78649
TEST 3: PASSED



---

## Problem 2: Fractional Parts of Cube Roots
Use numpy to calculate the constants listed at the bottom of page 11 of the Secure Hash Standard, following the steps below. These are the first 32 bits of the fractional parts of the cube roots of the first 64 prime numbers.
1. Write a function called primes(n) that generates the first n prime numbers.
2. Use the function to calculate the cube root of the first 64 primes.
3. For each cube root, extract the first thirty-two bits of the fractional part.
4. Display the result in hexadecimal.
5. Test the results against what is in the Secure Hash Standard.

SHA-256 uses a fixed table of 64 32-bit constants (usually referred to as `K[0..63]`) during the compression rounds. Rather than picking these values arbitrarily, the Secure Hash Standard defines them as:

- take the first 64 prime numbers,
- compute the cube root of each prime,
- take the fractional part only (everything after the decimal point),
- convert the first 32 bits of that fraction into a 32-bit word.

This approach produces a set of constants that look random but are fully reproducible. In this problem, I generate the constants directly from the definition and verify they match the official values listed in the standard.

### 2.1 Generating the first n primes

To compute the SHA-256 constants I first need the first `n` prime numbers. I use a simple primality test:

- handle small cases (`k < 2`, `k == 2`, even numbers),
- then test divisibility by odd numbers up to `sqrt(k)`.

This is not the most optimised prime generator, but it is easy to read, correct for this problem size (64 primes), and keeps the notebook self-contained.

In [2461]:
def is_prime(k):
    """
    Determine whether an integer k is a prime number.

    A prime number is greater than 1 and has no positive divisors
    other than 1 and itself. This function uses trial division
    up to sqrt(k), which is enough for the small values used here.

    Args:
        k: Integer to test for primality.

    Returns:
        bool: True if k is prime, False otherwise.
    """
    
    # Numbers less than 2 are not prime by definition
    if k < 2:
        return False
    
    # 2 is the only even prime number
    if k == 2:
        return True
    
    # Exclude all other even numbers
    if k % 2 == 0:
        return False
    
    # Check divisibility only up to sqrt(k)
    # If k has a factor greater than sqrt(k),
    # it must also have one smaller than sqrt(k)
    limit = int(k ** 0.5)
    
    # Check only odd divisors (even ones already excluded)
    for d in range(3, limit + 1, 2):
        if k % d == 0:
            return False
    
    # If no divisors were found, k is prime
    return True


In [2462]:
def primes(n):
    """
    Generate the first n prime numbers.

    Successive integers are tested for primality and collected
    until n prime numbers have been found.

    Args:
        n: Number of prime numbers to generate.

    Returns:
        list[int]: A list containing the first n prime numbers.
    """
    
    prime_list = []       # list to store found prime numbers
    candidate = 2         # first number to test for primality
    
    # Continue until we have collected n primes
    while len(prime_list) < n:
        
        # Check if the current candidate is prime
        if is_prime(candidate):
            prime_list.append(candidate)
        
        # Move on to the next number
        candidate += 1
    
    return prime_list


In [2463]:
def test_primes():
    print("Testing primes(n) function...")
    print("-" * 40)
    
    # Test case 1: First prime
    n = 1
    expected = [2]
    result = primes(n)
    
    print("Test 1:")
    print(" n =", n)
    print("Expected:", expected)
    print("Actual:  ", result)
    
    if expected == result:
        print("TEST 1: PASSED")
    else:
        print("TEST 1: FAILED")
    print()
    
    # Test case 2: First five primes
    n = 5
    expected = [2, 3, 5, 7, 11]
    result = primes(n)
    
    print("Test 2:")
    print(" n =", n)
    print("Expected:", expected)
    print("Actual:  ", result)
    
    if expected == result:
        print("TEST 2: PASSED")
    else:
        print("TEST 2: FAILED")
    print()
    
    # Test case 3: First ten primes
    n = 10
    expected = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
    result = primes(n)
    
    print("Test 3:")
    print(" n =", n)
    print("Expected:", expected)
    print("Actual:  ", result)
    
    if expected == result:
        print("TEST 3: PASSED")
    else:
        print("TEST 3: FAILED")
    print()


In [2464]:
test_primes()

Testing primes(n) function...
----------------------------------------
Test 1:
 n = 1
Expected: [2]
Actual:   [2]
TEST 1: PASSED

Test 2:
 n = 5
Expected: [2, 3, 5, 7, 11]
Actual:   [2, 3, 5, 7, 11]
TEST 2: PASSED

Test 3:
 n = 10
Expected: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
Actual:   [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
TEST 3: PASSED



### 2.2 Computing the SHA-256 round constants

Once the primes are generated, the constants are computed from the fractional part of each cube root.

For each prime `p`:

1. Compute the cube root using NumPy (`np.cbrt`).
2. Extract the fractional part by subtracting `floor(root)`.
3. Scale the fraction by `2**32` to shift the first 32 fractional bits into the integer range.
4. Take `floor(...)` and cast to `np.uint32` to get a 32-bit word.
5. Format the result as an 8-digit hexadecimal value for comparison against the standard.

The output is displayed in hexadecimal because the Secure Hash Standard lists the constants in hex form.

In [2465]:
def cube_roots(n):
    """
    Compute the SHA-256 round constants (K values) as defined
    in the Secure Hash Standard (FIPS 180-4).

    For each of the first n prime numbers:
    - compute the cube root,
    - extract the fractional part,
    - take the first 32 bits of that fraction.

    Args:
        n: Number of constants to generate (64 for SHA-256).

    Returns:
        list[numpy.uint32]: List of 32-bit constants derived from
        the fractional parts of cube roots of the first n primes.
    """
    prime_numbers = primes(n)
    ks = []

    for p in prime_numbers:
        # Use float cube root
        r = np.cbrt(np.float64(p))

        # Fractional part: r - floor(r)
        frac = r - np.floor(r)

        # Convert first 32 fractional bits into an integer:
        # floor(frac * 2^32)
        word = np.uint32(np.floor(frac * (2**32)))

        ks.append(word)

    return ks

In [2466]:
def to_hex32(words):
    """
    Convert a list of 32-bit words to hexadecimal strings.

    Each value is formatted as an 8-character, zero-padded,
    lowercase hexadecimal string, matching the representation
    used in the Secure Hash Standard.

    Args:
        words: Iterable of uint32 values.

    Returns:
        list[str]: List of 8-digit hexadecimal strings.
    """
    return [f"{int(w):08x}" for w in words]

Ks = cube_roots(64)
print(to_hex32(Ks)[:8])


['428a2f98', '71374491', 'b5c0fbcf', 'e9b5dba5', '3956c25b', '59f111f1', '923f82a4', 'ab1c5ed5']


### 2.3 Verifying against the Secure Hash Standard

The standard provides the expected `K[0..63]` values. I include those values directly in the notebook and compare them to the computed result.

The test prints:
- sanity check of the first 8 values,
- then a full check across all 64 constants.

If there is any mismatch, the test reports the index of the first difference to make debugging easier.

In [2467]:
EXPECTED_KS = [
    "428a2f98","71374491","b5c0fbcf","e9b5dba5","3956c25b","59f111f1","923f82a4","ab1c5ed5",
    "d807aa98","12835b01","243185be","550c7dc3","72be5d74","80deb1fe","9bdc06a7","c19bf174",
    "e49b69c1","efbe4786","0fc19dc6","240ca1cc","2de92c6f","4a7484aa","5cb0a9dc","76f988da",
    "983e5152","a831c66d","b00327c8","bf597fc7","c6e00bf3","d5a79147","06ca6351","14292967",
    "27b70a85","2e1b2138","4d2c6dfc","53380d13","650a7354","766a0abb","81c2c92e","92722c85",
    "a2bfe8a1","a81a664b","c24b8b70","c76c51a3","d192e819","d6990624","f40e3585","106aa070",
    "19a4c116","1e376c08","2748774c","34b0bcb5","391c0cb3","4ed8aa4a","5b9cca4f","682e6ff3",
    "748f82ee","78a5636f","84c87814","8cc70208","90befffa","a4506ceb","bef9a3f7","c67178f2"
]


In [2468]:
def test_cube_roots():
    print("Testing Problem 2 constants (cube roots of first 64 primes)...\n")

    computed = to_hex32(cube_roots(64))

    # Quick check: first 8
    print("First 8 computed: ", computed[:8])
    print("First 8 expected: ", EXPECTED_KS[:8])
    print()

    # Full check
    all_match = (computed == EXPECTED_KS)

    if all_match:
        print("FULL TEST: PASSED (all 64 constants match the standard)")
    else:
        print("FULL TEST: FAILED")
        # Print first mismatch to debug
        for i, (c, e) in enumerate(zip(computed, EXPECTED_KS)):
            if c != e:
                print(f"First mismatch at index {i}: computed={c}, expected={e}")
                break

    print()


In [2469]:
test_cube_roots()

Testing Problem 2 constants (cube roots of first 64 primes)...

First 8 computed:  ['428a2f98', '71374491', 'b5c0fbcf', 'e9b5dba5', '3956c25b', '59f111f1', '923f82a4', 'ab1c5ed5']
First 8 expected:  ['428a2f98', '71374491', 'b5c0fbcf', 'e9b5dba5', '3956c25b', '59f111f1', '923f82a4', 'ab1c5ed5']

FULL TEST: PASSED (all 64 constants match the standard)



---

## Problem 3: Padding
Write a generator function block_parse(msg) that processes messages according to section 5.1.1 and 5.2.1 of the Secure Hash Standard. The function should accept a bytes object called msg. At each iteration, it should yield the next 512-bit block of msg as a bytes object. Ensure that the final block (or final two blocks) include(s) the required padding of msg as specified in the standard. Test the generator with messages of different lengths to confirm proper padding and block output.


SHA-256 processes input messages in fixed-size 512-bit blocks (64 bytes). Real messages are rarely an exact multiple of 64 bytes, so the standard defines a padding scheme that:

- ensures the final padded message length is a multiple of 512 bits,
- ensures the original message length is still recoverable from the padded form,
- prevents ambiguity, as two different messages should not end up with the same padded bytes.

**Padding rules**  
Given an input message `msg`:

1. Append a single 1 bit to the message (in bytes, this is `0x80`, i.e. `10000000`).
2. Append 0 bits until the total length is 64 bits short of a full block (message length is 56 bytes mod 64).
3. Append the original message length as a 64-bit big-endian integer.

After this, the message length will be a multiple of 64 bytes and can be processed block by block.

In [2470]:
def block_parse(msg):
    """
    Yield 512-bit (64-byte) blocks from a message using SHA-256 padding.

    The message is padded according to FIPS 180-4:
    - append a single '1' bit,
    - append '0' bits until the length is 56 bytes mod 64,
    - append the original message length as a 64-bit big-endian integer.

    Args:
        msg: Input message as a bytes (or bytearray) object.

    Yields:
        bytes: The next 64-byte block of the padded message.
    """
    if not isinstance(msg, (bytes, bytearray)):
        raise TypeError("Invalid input - must be a byte object")

    # get original message length in bits
    bit_len = len(msg) * 8

    # append the '1' bit
    padded = msg + b"\x80"

    # append zero bytes until length is 56 mod 64.
    # 56 bytes = 448 bits, which leaves 8 bytes, or 64 bits, for the length field
    while (len(padded) % 64) != 56:
        padded += b"\x00"

    # append the original message length in bits
    padded += bit_len.to_bytes(8, byteorder="big")

    # parse into 512-bit (64-byte) blocks and yield each one.
    for i in range(0, len(padded), 64):
        yield padded[i:i+64]


#### Padding Tests

The test cases are chosen to hit the padding edge cases described in the standard:

- **Empty message**: still produces a full block after padding, with a length field of 0.
- **Short message ("abc")**: small input to confirm the general case.
- **55 bytes**: the largest message that still fits into one padded block (because `0x80` + 8-byte length still fit).
- **56 bytes**: triggers the “two block” padding case because there is no room left for the 8-byte length field in the first block.
- **64 bytes**: already exactly one full block, so padding adds an entire extra block.

For each case, the test checks:
- every yielded block is exactly 64 bytes,
- the last 8 bytes of the final block match the original message length (in bits).

In [2471]:
def test_block_parse():
    print("Testing block_parse(msg)...\n")

    tests = [
        (b"", "empty message"),
        (b"abc", "short message"),
        (b"a" * 55, "55 bytes (still 1 block after padding)"),
        (b"a" * 56, "56 bytes (forces 2 blocks after padding)"),
        (b"a" * 64, "64 bytes (full block; padding adds another)"),
    ]

    for msg, label in tests:
        blocks = list(block_parse(msg))
        print(f"{label}:")
        print("  original bytes:", len(msg))
        print("  blocks yielded:", len(blocks))
        print("  total padded bytes:", sum(len(b) for b in blocks))

        # show last 8 bytes of the final block
        last_block = blocks[-1]
        length_field = last_block[-8:]
        expected_len_bytes = (len(msg) * 8).to_bytes(8, "big")

        print("  length field (hex):", length_field.hex())
        print("  expected     (hex):", expected_len_bytes.hex())

        if length_field == expected_len_bytes:
            print("  LENGTH FIELD: PASSED")
        else:
            print("  LENGTH FIELD: FAILED")

        # every yielded block must be exactly 64 bytes
        all_64 = all(len(b) == 64 for b in blocks)
        print("  all blocks 64 bytes:", all_64)
        print()


In [2472]:
test_block_parse()

Testing block_parse(msg)...

empty message:
  original bytes: 0
  blocks yielded: 1
  total padded bytes: 64
  length field (hex): 0000000000000000
  expected     (hex): 0000000000000000
  LENGTH FIELD: PASSED
  all blocks 64 bytes: True

short message:
  original bytes: 3
  blocks yielded: 1
  total padded bytes: 64
  length field (hex): 0000000000000018
  expected     (hex): 0000000000000018
  LENGTH FIELD: PASSED
  all blocks 64 bytes: True

55 bytes (still 1 block after padding):
  original bytes: 55
  blocks yielded: 1
  total padded bytes: 64
  length field (hex): 00000000000001b8
  expected     (hex): 00000000000001b8
  LENGTH FIELD: PASSED
  all blocks 64 bytes: True

56 bytes (forces 2 blocks after padding):
  original bytes: 56
  blocks yielded: 2
  total padded bytes: 128
  length field (hex): 00000000000001c0
  expected     (hex): 00000000000001c0
  LENGTH FIELD: PASSED
  all blocks 64 bytes: True

64 bytes (full block; padding adds another):
  original bytes: 64
  blocks y

---

## Problem 4: Hashes (Compression)
Write a function hash(current, block) that calculates the next hash value given the current hash value and the next message block according to section 6.2.2 SHA-256 Hash Computation on page 22 of the Secure Hash Standard.


After padding, SHA-256 processes the message one 512-bit block at a time. Each block updates an internal state of eight 32-bit words (often written as H0..H7). This block-by-block update is the “compression function”.

At a high level, for each 512-bit block:

1. Split the block into 16 words (32-bit big-endian values).
2. Expand those 16 words into a 64-word message schedule `W[0..63]` using the small sigma functions (`sigma0`, `sigma1`).
3. Initialise eight working variables (`a`..`h`) from the current hash state.
4. Run 64 rounds. Each round mixes:
   - the current working variables,
   - the message schedule word `W[t]`,
   - the round constant `K[t]`,
   - and the logical functions (`Ch`, `Maj`, `Sigma0`, `Sigma1`).
5. Add the final working variables back into the current state to produce the next state.

This section implements those steps so that the final digest matches Python’s built-in `hashlib.sha256()` for the same input.

In [2473]:
def block_words(block):
    """
    Convert a 512-bit (64-byte) message block into 16 32-bit words.

    SHA-256 interprets each block as sixteen 32-bit big-endian integers.
    These values form the starting point for the message schedule.

    Args:
        block: A 64-byte bytes object.

    Returns:
        list[numpy.uint32]: 16 words parsed from the block in big-endian order.

    Raises:
        ValueError: If block is not exactly 64 bytes.
    """
    if len(block) != 64:
        raise ValueError("block must be exactly 64 bytes (512 bits)")

    words = []
    for i in range(0, 64, 4):
        w = int.from_bytes(block[i:i+4], byteorder="big")
        words.append(np.uint32(w))
    return words

In [2474]:
def message_schedule(block):
    """
    Build the SHA-256 message schedule W[0..63] from one 512-bit block.

    - W[0..15] are the 16 words directly parsed from the block.
    - W[16..63] are computed using earlier schedule values and the
      sigma0/sigma1 functions (the 'small sigma' functions).

    Args:
        block: A 64-byte bytes object (one padded 512-bit block)

    Returns:
        list[numpy.uint32]: A list of 64 uint32 words forming the schedule.

    Raises:
        ValueError: If block is not exactly 64 bytes.
    """
    if len(block) != 64:
        raise ValueError("block must be 64 bytes")

    M = block_words(block)
    W = [np.uint32(0)] * 64

    for t in range(16):
        W[t] = np.uint32(M[t])
 
    for t in range(16, 64):
        W[t] = np.uint32(sigma1(W[t-2]) + W[t-7] + sigma0(W[t-15]) + W[t-16])

    return W


In [2475]:
def hash(current, block, K):
    """
    Apply one SHA-256 compression step.

    This function updates the current 256-bit state (8 x uint32 words)
    using one 512-bit message block and the 64 round constants K[0..63].

    - build the 64-word message schedule W,
    - initialise working variables a..h from the current state,
    - run 64 rounds of mixing,
    - add the working variables back into the state.

    Args:
        current: List of 8 uint32 values representing the current hash state.
        block:   64-byte message block (already padded via block_parse).
        K:       List of 64 uint32 SHA-256 round constants.

    Returns:
        list[numpy.uint32]: The next hash state (8 uint32 values).

    Raises:
        ValueError: If current is not length 8 or K is not length 64.
    """
    if len(current) != 8:
        raise ValueError("current must contain 8 uint32 words")
    if len(K) != 64:
        raise ValueError("K must contain 64 uint32 constants")

    # 1. message schedule
    W = message_schedule(block)

    # 2. initialise working variables a - h from the current hash value
    a, b, c, d, e, f, g, h = [np.uint32(x) for x in current]

    # 3 main compression loop (64 rounds)
    for t in range(64):
        # T1 = h + Σ1(e) + Ch(e,f,g) + K[t] + W[t]
        T1 = np.uint32(h + Sigma1(e) + Ch(e,f,g) + K[t] + W[t])

        # T2 = Σ0(a) + Maj(a,b,c)
        T2 = np.uint32(Sigma0(a) + Maj(a,b,c))

        # Update variables 
        h = g
        g = f
        f = e
        e = np.uint32(d + T1)
        d = c
        c = b
        b = a
        a = np.uint32(T1 + T2)

    # 4. find the next intermediate hash value: H(i) = H(i-1) + [a..h]
    H0, H1, H2, H3, H4, H5, H6, H7 = [np.uint32(x) for x in current]

    new_state = [
        np.uint32(H0 + a),
        np.uint32(H1 + b),
        np.uint32(H2 + c),
        np.uint32(H3 + d),
        np.uint32(H4 + e),
        np.uint32(H5 + f),
        np.uint32(H6 + g),
        np.uint32(H7 + h),
    ]

    return new_state


In [2476]:
# Initial hash values for SHA-256 (FIPS 180-4 5.3.3)
H0_256 = [
    np.uint32(0x6a09e667),
    np.uint32(0xbb67ae85),
    np.uint32(0x3c6ef372),
    np.uint32(0xa54ff53a),
    np.uint32(0x510e527f),
    np.uint32(0x9b05688c),
    np.uint32(0x1f83d9ab),
    np.uint32(0x5be0cd19),
]


In [2477]:
# SHA-256 round constants K[0..63] (from problem 2)
K = cube_roots(64)

#### Hash Tests

To verify the implementation, I compare the final digest produced by my code against `hashlib.sha256()`.

Each test:
- pads the input using `block_parse(msg)`,
- processes every 64-byte block through `hash(...)`,
- formats the final state as a 64-character hex digest,
- compares it to Python’s reference implementation.

The chosen messages cover:
- the empty message,
- a short ASCII message (`"abc"`),
- a simple phrase (`"hello world"`),
- and a longer well-known test string (quickly catches small mistakes in padding, scheduling, or round logic).

### Note on NumPy overflow warnings

When running the SHA-256 compression loop, NumPy may print warnings such as:

`RuntimeWarning: overflow encountered in scalar add`

These are expected. SHA-256 defines all additions on 32-bit words as **modulo \(2^{32}\)** arithmetic, meaning wraparound on overflow is part of the algorithm.  
Because the implementation uses `numpy.uint32` to enforce 32-bit behaviour, intermediate additions naturally overflow and wrap around - NumPy just warns about it.


In [2478]:
def test_hash():
    print("Testing hash(current, block, K)...\n")

    tests = [
        (b"", "empty message"),
        (b"abc", "short message"),
        (b"hello world", "simple string"),
        (b"The quick brown fox jumps over the lazy dog", "famous phrase"),
    ]

    for msg, label in tests:
        print(label + ":")

        # start from initial SHA-256 hash values
        current = H0_256.copy()

        # process each padded block
        blocks = list(block_parse(msg))
        print("  blocks processed:", len(blocks))

        for block in blocks:
            current = hash(current, block, K)

        # build final hex digest manually
        computed = ""
        for word in current:
            computed += f"{int(word):08x}"

        expected = hashlib.sha256(msg).hexdigest()

        print("  Expected:", expected)
        print("  Computed:", computed)

        if computed == expected:
            print("  TEST: PASSED")
        else:
            print("  TEST: FAILED")

        print()


In [2479]:
test_hash()

Testing hash(current, block, K)...

empty message:
  blocks processed: 1
  Expected: e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
  Computed: e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
  TEST: PASSED

short message:
  blocks processed: 1
  Expected: ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad
  Computed: ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad
  TEST: PASSED

simple string:
  blocks processed: 1
  Expected: b94d27b9934d3e08a52e52d7da7dabfac484efe37a5380ee9088f7ace2efcde9
  Computed: b94d27b9934d3e08a52e52d7da7dabfac484efe37a5380ee9088f7ace2efcde9
  TEST: PASSED

famous phrase:
  blocks processed: 1
  Expected: d7a8fbb307d7809469ca9abcb0082e4f8d5651e46d3cdb762d02d0bf37c9e592
  Computed: d7a8fbb307d7809469ca9abcb0082e4f8d5651e46d3cdb762d02d0bf37c9e592
  TEST: PASSED



/var/folders/lz/_q16ykf55pq_6q2p49yp3r040000gn/T/ipykernel_2823/2728654490.py:28: RuntimeWarning: overflow encountered in scalar add
  W[t] = np.uint32(sigma1(W[t-2]) + W[t-7] + sigma0(W[t-15]) + W[t-16])
/var/folders/lz/_q16ykf55pq_6q2p49yp3r040000gn/T/ipykernel_2823/1113803413.py:38: RuntimeWarning: overflow encountered in scalar add
  T1 = np.uint32(h + Sigma1(e) + Ch(e,f,g) + K[t] + W[t])
/var/folders/lz/_q16ykf55pq_6q2p49yp3r040000gn/T/ipykernel_2823/1113803413.py:41: RuntimeWarning: overflow encountered in scalar add
  T2 = np.uint32(Sigma0(a) + Maj(a,b,c))
/var/folders/lz/_q16ykf55pq_6q2p49yp3r040000gn/T/ipykernel_2823/1113803413.py:47: RuntimeWarning: overflow encountered in scalar add
  e = np.uint32(d + T1)
/var/folders/lz/_q16ykf55pq_6q2p49yp3r040000gn/T/ipykernel_2823/1113803413.py:51: RuntimeWarning: overflow encountered in scalar add
  a = np.uint32(T1 + T2)
/var/folders/lz/_q16ykf55pq_6q2p49yp3r040000gn/T/ipykernel_2823/1113803413.py:58: RuntimeWarning: overflow encounte

---

## Problem 5: Passwords
The following are the SHA-256 hashes of three common passwords that have been hashed using one pass of the SHA-256 algorithm. As strings, they were encoded using UTF-8. Determine the passwords and explain how you found them. Suggest ways in which the hashing of passwords could be improved to prevent the kind of attack you performed to find the passwords.
1. 5e884898da28047151d0e56f8dc6292773603d0d6aabbdd62a11ef721d1542d8
2. 873ac9ffea4dd04fa719e8920cd6938f0c23cd678af330939cff53c3d2855f34
3. b03ddf3ca2e714a6548e7495e2a03f5e824eaac9837cd7f159c67b90fb4b7342


In this problem I’m given three SHA-256 hashes that were produced by hashing common passwords once (UTF-8 encoded). The goal is to recover the original passwords and then explain why this was possible.

Because these are described as *common* passwords, a full brute-force search isn’t necessary. Instead, I used a dictionary attack:

- load a wordlist of commonly used passwords from a text file (Source: First 2000 entries from https://github.com/danielmiessler/SecLists/blob/master/Passwords/Common-Credentials/100k-most-used-passwords-NCSC.txt),
- hash each candidate with SHA-256,
- compare the resulting hex digest to the target hashes,
- stop early if all targets have been matched.

This works because SHA-256 is deterministic: the same input string always produces the same hash output. If the original password is in the wordlist, it will be found.

### 5.1 Loading a password list

To run the dictionary attack, I first load passwords from a file (`2000-most-used-passwords.txt`).  
Each line in the file is treated as one candidate password. Blank lines are ignored.

I print the number of loaded passwords and the first few entries as a quick check that:
- the file was found,
- it was read correctly,
- the list looks like the expected wordlist.

In [2480]:
def load_password_list(path):
    """
    Load candidate passwords from a text file.

    Each non-empty line is treated as one candidate password.
    Leading/trailing whitespace is removed. The file is read using UTF-8.

    Args:
        path (str): Path to a password wordlist file.

    Returns:
        list[str]: List of candidate passwords in the order they appear in the file.
    """
    passwords = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            pw = line.strip()
            if pw:  # skip empty lines
                passwords.append(pw)
    return passwords

WORDLIST_PATH = "2000-most-used-passwords.txt"
COMMON_PASSWORDS = load_password_list(WORDLIST_PATH)

print("Loaded passwords:", len(COMMON_PASSWORDS))
print("First 10:", COMMON_PASSWORDS[:10])


Loaded passwords: 2000
First 10: ['123456', '123456789', 'qwerty', 'password', '111111', '12345678', 'abc123', '1234567', 'password1', '12345']


In [2481]:
# Target SHA-256 hashes from Problem 5
TARGET_HASHES = [
    "5e884898da28047151d0e56f8dc6292773603d0d6aabbdd62a11ef721d1542d8",
    "873ac9ffea4dd04fa719e8920cd6938f0c23cd678af330939cff53c3d2855f34",
    "b03ddf3ca2e714a6548e7495e2a03f5e824eaac9837cd7f159c67b90fb4b7342",
]


### 5.2 Dictionary attack

The actual cracking step is just “hash and compare”:

1. Convert each candidate password to bytes using UTF-8.
2. Compute its SHA-256 digest.
3. Check if the digest matches any of the target hashes.

I convert the list of target hashes to a set so checks are fast.  
When a match is found, the password is recorded and printed immediately. The loop stops early if all target hashes have been recovered.

This kind of attack works well here because the passwords are likely to be in a common-password list, and because the hashes are unsalted (there’s nothing unique added per password).

In [2482]:
def crack_hashes_with_dictionary(target_hashes, candidates, show_progress=False):
    """
    Attempt to recover plaintext passwords using a dictionary attack.

    Each candidate password is UTF-8 encoded, hashed once with SHA-256,
    and compared to the target hash list. Matching passwords are recorded.

    Args:
        target_hashes (list[str]): SHA-256 hex digests to crack.
        candidates (list[str]): Candidate plaintext passwords (wordlist).
        show_progress (bool): If True, prints progress updates periodically.

    Returns:
        dict[str, str]: Mapping of {target_hash: recovered_password} for any matches found.
    """
    target_set = set(target_hashes)
    found = {}  # target_hash -> password

    for i, pwd in enumerate(candidates, start=1):
        h = hashlib.sha256(pwd.encode("utf-8")).hexdigest()

        if h in target_set and h not in found:
            found[h] = pwd
            print(f"FOUND: {h}  ->  '{pwd}'")

            # stop early if found everything
            if len(found) == len(target_set):
                print("All hashes found — stopping early.")
                break

        if show_progress and i % 250 == 0:
            print(f"Checked {i} candidates...")

    return found


### 5.3 How to improve password hashing

The main issue is that SHA-256 may be too fast for password storage. Fast hashes make large-scale guessing attacks very cheap.

Better practice is:
- Use a random salt per password (stored alongside the hash). This prevents precomputed attacks and ensures identical passwords don’t share the same hash.
- Use a slow, password-specific hashing scheme such as Argon2, bcrypt, scrypt, or PBKDF2. These are designed to be expensive to compute and harder to accelerate on GPUs/ASICs.
- Optionally add a server-side pepper (a secret value not stored in the database) to make database-only leaks harder to exploit.

With salting and a slow password hash, a dictionary attack can still work, but it becomes dramatically slower and much less practical at scale.


#### Password Cracking Tests

The password cracking code is tested by attempting to recover all three provided SHA-256 hashes using a dictionary attack.

The test performs the following steps:
- loads a list of common passwords from a text file,
- hashes each candidate using SHA-256,
- compares each hash against the target values,
- records and prints any successful matches.

This test demonstrates why storing passwords as plain SHA-256 hashes is insecure and motivates the need for stronger password hashing techniques.

In [2483]:
def test_crack_hashes_with_dictionary():
    print("Testing Problem 5: Password cracking (dictionary attack)\n")

    found = crack_hashes_with_dictionary(
        TARGET_HASHES,
        COMMON_PASSWORDS,
        show_progress=False
    )

    all_found = True

    print("\nResults:")
    for idx, h in enumerate(TARGET_HASHES, start=1):
        pw = found.get(h)
        if pw:
            print(f"{idx}. {h} -> '{pw}'")
        else:
            print(f"{idx}. {h} -> NOT FOUND")
            all_found = False

    print()
    if all_found:
        print("TEST PASSED: All passwords successfully recovered")
    else:
        print("TEST FAILED: One or more passwords were not recovered")


In [2484]:
test_crack_hashes_with_dictionary()

Testing Problem 5: Password cracking (dictionary attack)

FOUND: 5e884898da28047151d0e56f8dc6292773603d0d6aabbdd62a11ef721d1542d8  ->  'password'
FOUND: 873ac9ffea4dd04fa719e8920cd6938f0c23cd678af330939cff53c3d2855f34  ->  'cheese'
FOUND: b03ddf3ca2e714a6548e7495e2a03f5e824eaac9837cd7f159c67b90fb4b7342  ->  'P@ssw0rd'
All hashes found — stopping early.

Results:
1. 5e884898da28047151d0e56f8dc6292773603d0d6aabbdd62a11ef721d1542d8 -> 'password'
2. 873ac9ffea4dd04fa719e8920cd6938f0c23cd678af330939cff53c3d2855f34 -> 'cheese'
3. b03ddf3ca2e714a6548e7495e2a03f5e824eaac9837cd7f159c67b90fb4b7342 -> 'P@ssw0rd'

TEST PASSED: All passwords successfully recovered


---

## End